# Week 55, Day 2: Bit tricks

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w055d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
x = 0b101100
print(bin(x - 1))
print(bin(x & (x - 1)))
print(bin(x & -x))
print(bin(x | (x - 1)))

In [ ]:
def count_bits(x):
    count = 0
    while x:
        x &= x - 1
        count += 1
    return count

def is_power_of_two(x):
    return x > 0 and x & (x - 1) == 0

def members(mask):
    out = []
    while mask:
        low = mask & -mask
        out.append(low.bit_length() - 1)
        mask ^= low
    return out

print(count_bits(0b101100), [n for n in range(1, 70) if is_power_of_two(n)], members(0b101100))

In [ ]:
from functools import reduce
from operator import xor

readings = [4, 9, 2, 4, 7, 2, 9]
print(reduce(xor, readings))

def two_lonely(values):
    both = reduce(xor, values)
    bit = both & -both
    a = reduce(xor, (v for v in values if v & bit), 0)
    return sorted((a, both ^ a))

print(two_lonely([3, 8, 5, 3, 6, 8]), bin(0b1011 ^ 0b0110).count("1"))

In [ ]:
def same_size(n, k):
    out = []
    x = (1 << k) - 1
    while x < 1 << n:
        out.append(f"{x:0{n}b}")
        low = x & -x
        ripple = x + low
        x = ripple | ((x ^ ripple) >> 2) // low
    return out

print(same_size(5, 2))

In [ ]:
def gray(i):
    return i ^ (i >> 1)

def from_gray(g):
    i = 0
    while g:
        i ^= g
        g >>= 1
    return i

codes = [gray(i) for i in range(8)]
print([f"{c:03b}" for c in codes], [from_gray(c) for c in codes])

> **Key idea.** x & (x − 1) clears the lowest set bit and x & −x isolates it: count bits by clearing them, test powers of two with x > 0 and x & (x − 1) == 0, and walk a mask's members with low = x & −x. XOR cancels pairs, finding lonely or missing values in O(1) memory, and the bit count of a ^ b is the Hamming distance. Gosper's hack steps through all masks with k bits set in increasing order. Gray code i ^ (i >> 1) changes one bit between neighbours. Rotation and reversal need a chosen width.

### Exercise: Lowest bits

Write `count_bits(x)` with the `x & (x - 1)` loop, `is_power_of_two(x)`, and `next_power_of_two(x)`: the smallest power of two that is at least x, for x ≥ 1 (hint: `bit_length` of x − 1). Use no strings or `bin`.

In [ ]:
def count_bits(x):
    ___

def is_power_of_two(x):
    ___

def next_power_of_two(x):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_low():\n    \"lowest bits\"\n    assert count_bits(0) == 0 and count_bits(0b101100) == 3 and count_bits((1 << 200) - 1) == 200\n    assert [n for n in range(-2, 20) if is_power_of_two(n)] == [1, 2, 4, 8, 16]\n    assert [next_power_of_two(n) for n in (1, 2, 3, 5, 8, 9, 1000)] == [1, 2, 4, 8, 8, 16, 1024]\n    rng = random.Random(1)\n    for _ in range(300):\n        n = rng.randint(1, 1 << 70)\n        p = next_power_of_two(n)\n        assert is_power_of_two(p) and p >= n and p // 2 < n and count_bits(n) == bin(n).count(\"1\")\n    assert \"bin(\" not in source and \"str(\" not in source")

### Exercise: Without a partner

Write `lonely(values)`: the one value appearing an **odd** number of times (all others appear an even number of times), with XOR. Then `missing(values, n)`: the one number from 0 to n not in the list (which has the other n numbers), with XOR and no sum. Then `two_lonely(values)`: the two values appearing once (all others twice), sorted.

In [ ]:
def lonely(values):
    ___

def missing(values, n):
    ___

def two_lonely(values):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_lonely():\n    \"XOR\"\n    assert lonely([4, 9, 2, 4, 7, 2, 9]) == 7 and lonely([5]) == 5 and lonely([1, 1, 1]) == 1\n    assert missing([3, 0, 1], 3) == 2 and missing([], 0) == 0\n    assert two_lonely([3, 8, 5, 3, 6, 8]) == [5, 6] and two_lonely([0, 7]) == [0, 7]\n    rng = random.Random(2)\n    for _ in range(100):\n        pairs = rng.sample(range(1000), 20)\n        a, b = rng.sample(range(1000, 2000), 2)\n        vals = pairs * 2 + [a, b]\n        rng.shuffle(vals)\n        assert two_lonely(vals) == sorted([a, b]) and lonely(pairs * 2 + [a]) == a\n        n = rng.randint(1, 50)\n        nums = list(range(n + 1))\n        gone = nums.pop(rng.randrange(n + 1))\n        rng.shuffle(nums)\n        assert missing(nums, n) == gone\n    assert \"sum(\" not in source")

### Exercise: Gosper's hack

Write `masks_of_size(n, k)`: a list of every n-bit mask with exactly k bits set, in increasing order, using Gosper's hack (handle k = 0 and k > n). Then `choose_items(items, k)`: every k-item selection from a list, as lists in the order the masks give them (bit i means `items[i]`).

In [ ]:
def masks_of_size(n, k):
    ___

def choose_items(items, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools\nfrom math import comb\ndef test_gosper():\n    \"same size\"\n    assert masks_of_size(4, 2) == [0b0011, 0b0101, 0b0110, 0b1001, 0b1010, 0b1100]\n    assert masks_of_size(3, 0) == [0] and masks_of_size(2, 3) == [] and masks_of_size(3, 3) == [7]\n    for n in range(0, 11):\n        for k in range(0, n + 1):\n            ms = masks_of_size(n, k)\n            assert len(ms) == comb(n, k) and ms == sorted(ms) and all(bin(m).count(\"1\") == k for m in ms)\n            assert all(m < 1 << n for m in ms)\n    assert sorted(map(sorted, choose_items(list(\"abcde\"), 3))) == sorted(map(sorted, itertools.combinations(\"abcde\", 3)))\n    assert choose_items([\"x\", \"y\", \"z\"], 2)[0] == [\"x\", \"y\"]\n    assert \"combinations\" not in source")

### Exercise: One bit at a time

Write `gray(i)`, `from_gray(g)` and `gray_sequence(n)`: the list of all 2ⁿ n-bit Gray codes in order. Then `toggle_order(items)`: a list of moves visiting every subset of `items` by adding or removing **one** item each time, starting from the empty set: each move is `("add", item)` or `("remove", item)`, from consecutive Gray codes (the changed bit i means `items[i]`).

In [ ]:
def gray(i):
    ___

def from_gray(g):
    ___

def gray_sequence(n):
    ___

def toggle_order(items):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_gray():\n    \"Gray codes\"\n    assert gray_sequence(3) == [0, 1, 3, 2, 6, 7, 5, 4] and gray_sequence(0) == [0]\n    for n in range(1, 11):\n        seq = gray_sequence(n)\n        assert sorted(seq) == list(range(1 << n))\n        assert all(bin(a ^ b).count(\"1\") == 1 for a, b in zip(seq, seq[1:]))\n        assert all(from_gray(gray(i)) == i for i in range(1 << n))\n    moves = toggle_order([\"a\", \"b\", \"c\"])\n    assert len(moves) == 7 and moves[:3] == [(\"add\", \"a\"), (\"add\", \"b\"), (\"remove\", \"a\")]\n    held, seen = set(), {frozenset()}\n    for action, item in moves:\n        (held.add if action == \"add\" else held.remove)(item)\n        seen.add(frozenset(held))\n    assert len(seen) == 8")

### Exercise: Turning bits round

Write `rotate_left(x, k, width)` and `rotate_right(x, k, width)` for a `width`-bit value (k may be larger than width), `reverse_bits(x, width)` without strings, and `hamming(a, b)`: how many bits differ.

In [ ]:
def rotate_left(x, k, width):
    ___

def rotate_right(x, k, width):
    ___

def reverse_bits(x, width):
    ___

def hamming(a, b):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_rotate():\n    \"rotations\"\n    assert rotate_left(0b1001, 1, 4) == 0b0011 and rotate_right(0b1001, 1, 4) == 0b1100 and rotate_left(0b1, 9, 8) == 0b10\n    assert reverse_bits(0b0001, 4) == 0b1000 and reverse_bits(0b1101, 4) == 0b1011 and reverse_bits(1, 32) == 1 << 31\n    assert hamming(0b1011, 0b0110) == 3 and hamming(7, 7) == 0\n    rng = random.Random(3)\n    for _ in range(300):\n        w = rng.randint(1, 64)\n        x = rng.randrange(1 << w)\n        k = rng.randint(0, 200)\n        assert rotate_right(rotate_left(x, k, w), k, w) == x and rotate_left(x, k, w) < 1 << w\n        assert reverse_bits(reverse_bits(x, w), w) == x and reverse_bits(x, w) == int(f\"{x:0{w}b}\"[::-1], 2)\n    assert \"str(\" not in source.split(\"def reverse_bits\")[1].split(\"def hamming\")[0]\n    assert \"[::-1]\" not in source")

### Bug hunt: Zero is not a power of two

`0 & (0 - 1)` is 0, so this test says 0 is a power of two and prints `[0, 1, 2, 4, 8]`. Require x to be positive so it prints `[1, 2, 4, 8]`.

In [ ]:
print([x for x in range(10) if x & (x - 1) == 0])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"positive\"\n    expect_output(\"[1, 2, 4, 8]\")")

### Bug hunt: Positions one too high

`bit_length` counts the bits needed to write a number, so the lowest set bit `0b100` has bit length 3 but sits at position 2. This walk through the members of `0b101100` forgets to subtract 1 and prints `[3, 4, 6]`. Fix it so it prints `[2, 3, 5]`.

In [ ]:
mask, out = 0b101100, []
while mask:
    low = mask & -mask
    out.append(low.bit_length())
    mask ^= low
print(out)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"positions from 0\"\n    expect_output(\"[2, 3, 5]\")")

### Bug hunt: Gray code the wrong way

`i ^ (i << 1)` isn't a Gray code: neighbours can differ in several bits, and the values run past n bits. This prints `False`; use `i >> 1` so every pair of neighbours differs in exactly one bit and it prints `True`.

In [ ]:
codes = [i ^ (i << 1) for i in range(16)]
print(all(bin(a ^ b).count("1") == 1 for a, b in zip(codes, codes[1:])))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"shift right\"\n    expect_output(\"True\")")

### Puzzle: Gosper's hack

Arrange the lines that list every 5-bit mask with two bits set. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # 0b10100 & 0b10011 (x & (x - 1) for x = 0b10100), as a whole number
q = ___   # 40 & -40
r = ___   # gray(5)
s = ___   # the XOR of [6, 1, 6, 3, 1]
t = ___   # the Hamming distance between 10 and 5

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 16 and q == 8 and r == 7, \"p, q and r\"\n    assert s == 3 and t == 4, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the biggest XOR

Write `max_xor_pair(values)`: the largest `a ^ b` for two values (at different positions) in a list of non-negative whole numbers, in O(n × bits) rather than O(n²). Build the answer **one bit at a time from the top**: having decided the best possible top bits so far (`best`), ask whether `best | bit` is achievable. It is if some two values' top-bit prefixes XOR to it, which you can test with a set of prefixes: for each prefix p, is `p ^ candidate` also in the set? (A ^ B = C exactly when A ^ C = B.)

Then `max_xor_with(values, queries)`: for each query q, the largest `q ^ v` for v in the list, using a **binary trie** built from nested two-item lists (Week 47's tries, with the alphabet {0, 1}): walk down from the top bit, preferring the child with the opposite bit.

In [ ]:
def max_xor_pair(values):
    ___

def max_xor_with(values, queries):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef test_xor():\n    \"greedy bits\"\n    assert max_xor_pair([3, 10, 5, 25, 2, 8]) == 28 and max_xor_pair([7, 7]) == 0 and max_xor_pair([0, 1]) == 1\n    assert max_xor_with([3, 10, 5, 25, 2, 8], [0, 7, 31]) == [25, 30, 29]\n    rng = random.Random(4)\n    for _ in range(100):\n        vals = [rng.randint(0, 1000) for _ in range(rng.randint(2, 12))]\n        assert max_xor_pair(vals) == max(a ^ b for i, a in enumerate(vals) for b in vals[i + 1:])\n        qs = [rng.randint(0, 1000) for _ in range(5)]\n        assert max_xor_with(vals, qs) == [max(q ^ v for v in vals) for q in qs]\n    big = [rng.randint(0, 1 << 30) for _ in range(20000)]\n    t0 = time.perf_counter()\n    max_xor_pair(big)\n    max_xor_with(big, big[:2000])\n    assert time.perf_counter() - t0 < 8")

## Warm-up

Subtracting 1 turns the lowest 1 into 0 and every 0 below it into 1, leaving the higher bits alone. So `x & (x - 1)` removes the lowest set bit, and `x & -x` keeps **only** it. Those two expressions are the core of today.

## The lowest set bit

Write x as `...a 1 0 0 0` (some higher bits a, then its lowest 1, then zeros). Then:

- `x - 1` is `...a 0 1 1 1`: the lowest 1 and the zeros below it flip.
- `x & (x - 1)` is `...a 0 0 0 0`: the **lowest set bit is cleared**.
- `-x` is `~x + 1`, which is `...~a 1 0 0 0`, so `x & -x` is `0...0 1 0 0 0`: the lowest set bit **alone**.

Uses:

- **Count bits** by clearing them one at a time: the loop runs once per **1**, not once per bit (Brian Kernighan popularised it; Peter Wegner published it in 1960).
- **Power of two**: x > 0 and `x & (x - 1) == 0` (exactly one bit set).
- **Walk the members** of a bitmask set: take `low = x & -x`, use it, then `x ^= low`.
- The position of the lowest set bit is `(x & -x).bit_length() - 1`.

## XOR cancels

Because `a ^ a == 0` and XOR doesn't care about order, XOR-ing a whole list cancels every value that appears an **even** number of times. So:

- In a list where every value appears twice except one, the XOR of everything is the **lonely** value. O(n) time, O(1) memory, no dictionary.
- The **missing** number from 0 to n: XOR everything from 0 to n with everything in the list.
- **Two** lonely values a and b: the XOR of everything is `a ^ b`, which isn't 0, so some bit differs between them. Take its lowest set bit, split the list by that bit, and XOR each half separately.
- **Hamming distance** (how many bits differ) is the bit count of `a ^ b`.

## Every set of size k

Last week's bitmask DPs looped over all 2ⁿ masks. Sometimes you need only the masks with exactly k bits set, in increasing order. **Gosper's hack** (Bill Gosper, in MIT's HAKMEM memo of 1972) steps from one to the next in a few operations:

1. `low = x & -x` (the lowest set bit);
2. `ripple = x + low` (carries through the lowest block of ones, setting the next 0 above it);
3. `ones = ((x ^ ripple) >> 2) // low` (the bits that changed, shifted back to the bottom: one fewer than the block's length);
4. the next mask is `ripple | ones`.

Start at `(1 << k) - 1` (the smallest) and stop when the mask reaches `1 << n`. It visits exactly C(n, k) masks, the same sets as `itertools.combinations`, in increasing numeric order.

(It needs k ≥ 1: for k = 0, `low` is 0. Handle that case separately.)

## Gray code

Counting in binary, 0111 to 1000 flips four bits at once. A **Gray code** orders all n-bit numbers so neighbours differ in **exactly one** bit (Frank Gray patented its use in 1953, for reading positions from rotating discs without glitches). The standard one is

`gray(i) = i ^ (i >> 1)`,

and to decode, XOR together all the right shifts of g: `i = g ^ (g >> 1) ^ (g >> 2) ^ ...`. Gray codes appear in rotary encoders, error correction, Karnaugh maps and in visiting every subset of a set by adding or removing **one** item at a time.

## Rotating and reversing

Python integers have no width, so to **rotate** (bits shifted off one end reappear at the other) you must choose one. For width w: `((x << k) | (x >> (w - k))) & ((1 << w) - 1)`, with k taken mod w. Rotations are used in hash functions and ciphers because they mix bits without losing any.

**Reversing** the bits of a w-bit number can be done one bit at a time, or with the string `f"{x:0{w}b}"[::-1]`. Both are fine in Python; the tricks with masks and swaps that you'll find in *Hacker's Delight* matter in languages where loops are expensive.

## Practice

## Quiz

## Daily challenge

## Before you go

Write the numbers 0 to 7 as 3-bit Gray codes and check by eye that each step changes one bit. Tomorrow: primes and modular arithmetic, the maths behind hashing and cryptography.